# HW2 — Part 2: FINN Synthesis

**Run this notebook on Lambda inside a CPU Slurm session, inside tmux.**  
See `README.md` section 1.8 for setup.

Synthesis times: W16A8 ~60 min, W2A1 ~30 min. Keep tmux attached.

In this notebook you will:
1. Synthesize W16A8 with FINN (watch all 19 steps)
2. Synthesize W2A1 with FINN
3. Inspect and compare resource utilization reports
4. Copy test data to the board
5. (Open-ended) Synthesize your custom model

## 0. Environment Check

In [1]:
import os, sys, subprocess

WORKDIR = os.path.expanduser('~/fpga_hw2')
os.chdir(WORKDIR)

# Ensure conda lib is on LD_LIBRARY_PATH so Vivado can find libtinfo.so.5
conda_lib = os.path.join(sys.prefix, 'lib')
os.environ['LD_LIBRARY_PATH'] = conda_lib + ':' + os.environ.get('LD_LIBRARY_PATH', '')

# Verify ONNX files from training exist
for f in ['model_w2a1.onnx', 'model_w16a8.onnx']:
    path = os.path.join(WORKDIR, f)
    assert os.path.exists(path), f'Missing {path} — run 01_training.ipynb first'
    print(f'Found: {f}  ({os.path.getsize(path)//1024} KB)')

# Verify Vivado is on PATH
r = subprocess.run(['vivado', '-version'], capture_output=True, text=True)
assert r.returncode == 0, 'Vivado not found — did you source the settings64.sh files?'
output = (r.stdout + r.stderr).strip().splitlines()
print('Vivado:', output[0] if output else 'OK')

# Verify FINN
import finn.builder.build_dataflow as build
print('FINN imported OK')

Found: model_w2a1.onnx  (790 KB)
Found: model_w16a8.onnx  (790 KB)
Vivado: Vivado v2022.1 (64-bit)


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


FINN imported OK


## 1. FINN Build Configuration (provided)

**What is FINN?** [FINN](https://github.com/Xilinx/finn) ([docs](https://finn.readthedocs.io/en/latest/)) is an open-source AMD/Xilinx framework that compiles quantized neural networks directly to streaming FPGA accelerators. Each network layer becomes a dedicated hardware block; the blocks are pipelined via AXI-Stream FIFOs into a fully custom dataflow accelerator with no CPU in the inference path. Background: [FINN-R paper](https://arxiv.org/abs/1809.04570) (Blott et al., ACM TRETS 2018).

FINN compiles the ONNX model through 19 steps:
- Steps 1–10: graph transformations, HW mapping, folding (fast, seconds)
- Step 12 (`hw_ipgen`): HLS synthesis — compiles each layer to Verilog (~5–10 min)
- Step 17 (`synthesize_bitfile`): Vivado place-and-route — produces the `.bit` file (~25–50 min)

**Folding config** controls parallelism:
- `SIMD`: how many input values are processed in parallel per cycle
- `PE`: how many output neurons are computed in parallel
- Higher SIMD/PE → faster inference, more LUT/FF usage

In [2]:
import json, sys
import finn.builder.build_dataflow_config as build_cfg

FINN_ROOT = os.path.expanduser('~/finn')
os.environ['FINN_ROOT']        = FINN_ROOT
os.environ['XILINX_VIVADO']   = '/home/yuval.mandel/Xilinx/Vivado/2022.1'
os.environ['XILINX_HLS']      = '/home/yuval.mandel/Xilinx/Vitis_HLS/2022.1'

_conda_lib = os.path.join(sys.prefix, 'lib')
os.environ['LD_LIBRARY_PATH'] = _conda_lib + ':' + os.environ.get('LD_LIBRARY_PATH', '')

def make_finn_cfg(onnx_path, output_dir, build_dir, folding_cfg_path, specialize_cfg_path):
    os.environ['FINN_BUILD_DIR'] = build_dir
    os.makedirs(build_dir, exist_ok=True)
    return build_cfg.DataflowBuildConfig(
        output_dir=output_dir,
        board='Pynq-Z2',
        shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
        synth_clk_period_ns=10.0,
        generate_outputs=[
            build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
            build_cfg.DataflowOutputType.BITFILE,
            build_cfg.DataflowOutputType.PYNQ_DRIVER,
            build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
        ],
        specialize_layers_config_file=specialize_cfg_path,
        folding_config_file=folding_cfg_path,
        auto_fifo_depths=False,
        verify_save_rtlsim_waveforms=False,
    )

def parse_resources(output_dir):
    path = os.path.join(output_dir, 'report', 'post_synth_resources.json')
    if not os.path.exists(path):
        print('Resource report not found — synthesis may not be complete.')
        return None
    with open(path) as f:
        return json.load(f)

def print_resources(resources, total={'LUT': 53200, 'FF': 106400, 'BRAM_36K': 140, 'DSP': 220}):
    if resources is None:
        return
    top = resources.get('(top)', {})
    print(f"{'Resource':<12} {'Used':>8} {'Available':>12} {'Utilization':>14}")
    print('-' * 50)
    for key, avail in total.items():
        used = top.get(key, top.get(key.replace('_36K', ''), 0))
        pct  = 100.0 * used / avail
        print(f"{key:<12} {used:>8,} {avail:>12,} {pct:>13.1f}%")

print('FINN build config utilities loaded.')

FINN build config utilities loaded.


---
## 2. Synthesize W16A8

Expected time: **~60 minutes**. Watch the 19 steps print below.

In [7]:
# SIMD=4 required: MW=3072, constraint SIMD >= MW/1024 = 3
# PE=1 required by BRAM: fc1 weights (3072×64×16b) already consume ~86 BRAM36K out of 140
# Increasing PE would split weight storage but add overhead that exceeds remaining BRAM headroom.
# When adapting for your open-ended model, start with PE=1 and increase only if BRAM allows.
folding_w16a8 = {
    "Defaults":   {},
    "MVAU_hls_0": {"SIMD": 4, "PE": 1},   # fc1: 3072→64, BRAM-limited to PE=1
    "MVAU_hls_1": {"SIMD": 4, "PE": 1},   # fc2: 64→64
}
specialize_w16a8 = {"Defaults": {"preferred_impl_style": ["rtl", "all"]}}

with open('folding_config_w16a8.json', 'w') as f: json.dump(folding_w16a8,   f, indent=2)
with open('specialize_w16a8.json',    'w') as f: json.dump(specialize_w16a8, f, indent=2)

for d in ['finn_build_w16a8_tmp', 'finn_output_w16a8']:
    p = os.path.join(WORKDIR, d)
    if os.path.exists(p):
        subprocess.run(['rm', '-rf', p], check=False)
        print(f'Removed stale {d}')

cfg_w16a8 = make_finn_cfg(
    onnx_path           = os.path.join(WORKDIR, 'model_w16a8.onnx'),
    output_dir          = os.path.join(WORKDIR, 'finn_output_w16a8'),
    build_dir           = os.path.join(WORKDIR, 'finn_build_w16a8_tmp'),
    folding_cfg_path    = os.path.join(WORKDIR, 'folding_config_w16a8.json'),
    specialize_cfg_path = os.path.join(WORKDIR, 'specialize_w16a8.json'),
)

print('Starting W16A8 FINN synthesis — ~60 minutes...')
build.build_dataflow_cfg(os.path.join(WORKDIR, 'model_w16a8.onnx'), cfg_w16a8)
print('\nW16A8 synthesis complete!')

Removed stale finn_build_w16a8_tmp
Removed stale finn_output_w16a8
Starting W16A8 FINN synthesis — ~60 minutes...
Building dataflow accelerator from /home/shahlaandrew/fpga_hw2/model_w16a8.onnx
Intermediate outputs will be generated in /home/shahlaandrew/fpga_hw2/finn_build_w16a8_tmp
Final outputs will be generated in /home/shahlaandrew/fpga_hw2/finn_output_w16a8
Build log is at /home/shahlaandrew/fpga_hw2/finn_output_w16a8/build_dataflow.log
Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]
Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
Running step: step_apply_folding_config [8/19]
Running step: step_minimize_bit_width [9/19]
Running step: step_generate_estimate_reports [10/19]
Running step: step_hw_codegen [11/19]
Running step: step_hw_ipgen [12/19]


bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)


Running step: step_set_fifo_depths [13/19]
Running step: step_create_stitched_ip [14/19]
Running step: step_measure_rtlsim_performance [15/19]
Running step: step_out_of_context_synthesis [16/19]
Running step: step_synthesize_bitfile [17/19]


bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (requi

Running step: step_make_pynq_driver [18/19]
Running step: step_deployment_package [19/19]
Completed successfully

W16A8 synthesis complete!


In [8]:
res_w16a8 = parse_resources(os.path.join(WORKDIR, 'finn_output_w16a8'))
print('W16A8 Resource Utilization:')
print_resources(res_w16a8)

W16A8 Resource Utilization:
Resource         Used    Available    Utilization
--------------------------------------------------
LUT            24,177       53,200          45.4%
FF             37,106      106,400          34.9%
BRAM_36K          132          140          94.3%
DSP                 9          220           4.1%


In [ ]:
W16A8 Resource Utilization:
Resource         Used    Available    Utilization
--------------------------------------------------
LUT            24,177       53,200          45.4%
FF             37,106      106,400          34.9%
BRAM_36K          132          140          94.3%
DSP                 9          220           4.1%

### Questions — W16A8 Resources

1. W16A8 uses a large fraction of available BRAM. Calculate the expected weight storage in bits for the fc1 layer and verify it matches the BRAM usage (one RAMB36 = 36 Kb). Does your measured result agree with the calculation?

**Your answer:**

1. the number of weights in fc1 is 3072*64 = 196,608, each weight is 16 bits so total bits is 3,145,728 bits, the expected BRAM usage is 3,145,728/36,000 ~ 88 VBRAM.

we are using 132 in our model, the difference in size comes from the fact that we also have fc2 and fc3 using the BRAM, and also FINN using BRAM. so its expected for the BRAM usage to be higher.

### Copy W16A8 deploy package to your laptop

Run the following **on your laptop** (not in this notebook):

```bash
scp -r <your-username>@132.68.39.159:~/fpga_hw2/finn_output_w16a8/deploy/ ./deploy_w16a8/
```

Then upload to the board:
```bash
scp -r ./deploy_w16a8/ xilinx@192.168.2.99:/home/xilinx/jupyter_notebooks/hw2/deploy_w16a8/
```

---
## 3. Synthesize W2A1

Expected time: **~30 minutes**. Watch the 19 steps print below.

In [3]:
# SIMD=4 required: first layer input width=3072, constraint SIMD >= MW/1024 = 3
folding_w2a1 = {
    "Defaults": {},
    "MVAU_hls_0": {"SIMD": 4, "PE": 1},
    "MVAU_hls_1": {"SIMD": 4, "PE": 1}
}
specialize_w2a1 = {"Defaults": {"preferred_impl_style": ["rtl", "all"]}}

with open('folding_config_w2a1.json', 'w')   as f: json.dump(folding_w2a1,   f, indent=2)
with open('specialize_w2a1.json',     'w')   as f: json.dump(specialize_w2a1, f, indent=2)

for d in ['finn_build_w2a1_tmp', 'finn_output_w2a1']:
    p = os.path.join(WORKDIR, d)
    if os.path.exists(p):
        subprocess.run(['rm', '-rf', p], check=False)
        print(f'Removed stale {d}')

cfg_w2a1 = make_finn_cfg(
    onnx_path        = os.path.join(WORKDIR, 'model_w2a1.onnx'),
    output_dir       = os.path.join(WORKDIR, 'finn_output_w2a1'),
    build_dir        = os.path.join(WORKDIR, 'finn_build_w2a1_tmp'),
    folding_cfg_path = os.path.join(WORKDIR, 'folding_config_w2a1.json'),
    specialize_cfg_path = os.path.join(WORKDIR, 'specialize_w2a1.json'),
)

print('Starting W2A1 FINN synthesis — ~30 minutes...')
build.build_dataflow_cfg(os.path.join(WORKDIR, 'model_w2a1.onnx'), cfg_w2a1)
print('\nW2A1 synthesis complete!')

Starting W2A1 FINN synthesis — ~30 minutes...
Building dataflow accelerator from /home/shahlaandrew/fpga_hw2/model_w2a1.onnx
Intermediate outputs will be generated in /home/shahlaandrew/fpga_hw2/finn_build_w2a1_tmp
Final outputs will be generated in /home/shahlaandrew/fpga_hw2/finn_output_w2a1
Build log is at /home/shahlaandrew/fpga_hw2/finn_output_w2a1/build_dataflow.log
Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]
Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
Running step: step_apply_folding_config [8/19]
Running step: step_minimize_bit_width [9/19]
Running step: step_generate_estimate_reports [10/19]
Running step: step_hw_codegen [11/19]
Running step: step_hw_ipgen [12/19]


bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)


Running step: step_set_fifo_depths [13/19]
Running step: step_create_stitched_ip [14/19]
Running step: step_measure_rtlsim_performance [15/19]
Running step: step_out_of_context_synthesis [16/19]
Running step: step_synthesize_bitfile [17/19]


bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by /bin/bash)
bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (required by bash)
/bin/bash: /home/shahlaandrew/miniforge3/envs/fpga_hw2/lib/libtinfo.so.6: no version information available (requi

Running step: step_make_pynq_driver [18/19]
Running step: step_deployment_package [19/19]
Completed successfully

W2A1 synthesis complete!


In [4]:
res_w2a1 = parse_resources(os.path.join(WORKDIR, 'finn_output_w2a1'))
print('W2A1 Resource Utilization:')
print_resources(res_w2a1)

W2A1 Resource Utilization:
Resource         Used    Available    Utilization
--------------------------------------------------
LUT             8,908       53,200          16.7%
FF             12,346      106,400          11.6%
BRAM_36K           17          140          12.1%
DSP                 1          220           0.5%


In [ ]:
W2A1 Resource Utilization:
Resource         Used    Available    Utilization
--------------------------------------------------
LUT             8,908       53,200          16.7%
FF             12,346      106,400          11.6%
BRAM_36K           17          140          12.1%
DSP                 1          220           0.5%

In [9]:
# Summary — paste these into 03_board_inference.ipynb cell-20
_total = {'LUT': 53200, 'FF': 106400, 'BRAM_36K': 140, 'DSP': 220}
def _pct(res, key):
    used = (res or {}).get('(top)', {}).get(key, 0)
    return round(100.0 * used / _total[key], 1)

print('=' * 60)
print('Paste these into 03_board_inference.ipynb cell-20:')
print('=' * 60)
print('resources = {')
for key in ['LUT', 'FF', 'BRAM_36K', 'DSP']:
    key_q = "'" + key + "'"
    print(f"    {key_q:<12s} {{'W2A1': {_pct(res_w2a1, key)}, 'W16A8': {_pct(res_w16a8, key)}}},")
print('}')

Paste these into 03_board_inference.ipynb cell-20:
resources = {
    'LUT'        {'W2A1': 16.7, 'W16A8': 45.4},
    'FF'         {'W2A1': 11.6, 'W16A8': 34.9},
    'BRAM_36K'   {'W2A1': 12.1, 'W16A8': 94.3},
    'DSP'        {'W2A1': 0.5, 'W16A8': 4.1},
}


In [ ]:
============================================================
Paste these into 03_board_inference.ipynb cell-20:
============================================================
resources = {
    'LUT'        {'W2A1': 16.7, 'W16A8': 45.4},
    'FF'         {'W2A1': 11.6, 'W16A8': 34.9},
    'BRAM_36K'   {'W2A1': 12.1, 'W16A8': 94.3},
    'DSP'        {'W2A1': 0.5, 'W16A8': 4.1},
}

### Questions — W2A1 Resources

1. W2A1 uses almost no DSPs. Why? What arithmetic operation replaces multiplication for binary networks?
2. W16A8 synthesis took significantly longer than W2A1. Why is that the case?

**Your answers:**

2. W16A8 is much larger, and more complex, it also uses 16 bit wieghts for all fc layers, processed with full fixed point arithmatic, W2A1 on the other hand uses cheap binary logic instead of heavy arithmatic, so layer computations are implemented via XNOR + popcount logic in LUTs and simple adders, with almost no DSP or BRAM structures, this makes it much smaller in LUTs and FFs, and much lighter in BRAM and DSP usage

### Copy W2A1 deploy package to your laptop

Run the following **on your laptop** (not in this notebook):

```bash
scp -r <your-username>@132.68.39.159:~/fpga_hw2/finn_output_w2a1/deploy/ ./deploy_w2a1/
```

Then upload to the board:
```bash
scp -r ./deploy_w2a1/ xilinx@192.168.2.99:/home/xilinx/jupyter_notebooks/hw2/deploy_w2a1/
```

---
## 4. Copy Test Data to the Board

Test data was generated at the end of `01_training.ipynb` (section 6). Copy the `.npy` files from Lambda to your laptop, then upload them to the board.

### Copy test data to your laptop

Test data was generated in `01_training.ipynb` section 6. Copy from Lambda:

```bash
scp <your-username>@132.68.39.159:~/fpga_hw2/test_images_w2a1_packed.npy ./
scp <your-username>@132.68.39.159:~/fpga_hw2/test_images_w16a8.npy       ./
scp <your-username>@132.68.39.159:~/fpga_hw2/test_labels.npy             ./
```

Then upload to the board:
```bash
scp test_images_w2a1_packed.npy test_images_w16a8.npy test_labels.npy \
    xilinx@192.168.2.99:/home/xilinx/jupyter_notebooks/hw2/
```